> **이 폴더는 [ECHO-TOFMS](../ECHO-TOFMS/) 분석 파이프라인에서 사용하는**
> **`echo_tof` 패키지의 내부 구현을 단계별로 설명합니다.**

# 01. Formula Enumeration

**목표**: `echo_tof.formula_enum` 모듈의 `FindElementalCompositions` 클래스를 이해한다.

질량분석기가 측정한 monoisotopic mass에서 가능한 모든 원소 조합(molecular formula)을
열거하는 것은 Formula Finder의 첫 번째 단계이다.  
원본 OS의 `Clearcore2.ElementsAndMolecules.FindElementalCompositions`를
Python으로 이식한 이 엔진은 **mass-constrained pruning**으로 탐색 공간을 효율적으로 줄인다.

## 환경 설정

In [ ]:
import os, sys
os.chdir(r'C:\Users\gogoc\AppData\Local\Temp\echo-tof-verify')
sys.path.insert(0, '.')

## 1. 핵심 데이터 구조: `ElementRange`

각 원소에 대해 탐색할 **최소~최대 원자 수**를 정의한다.
예를 들어 `max_composition="C50 H200 N10 O10 S5"`이면:

| 원소 | min | max |
|------|-----|-----|
| C    | 0   | 50  |
| H    | 0   | 200 |
| N    | 0   | 10  |
| O    | 0   | 10  |
| S    | 0   | 5   |

`ElementRange`는 현재 카운트(`_current_number`)와 해당 질량(`_current_mass`)을 추적하며,
`next()` 메서드로 odometer처럼 +1씩 증가한다.

In [ ]:
from echo_tof.formula_enum import ElementRange
from echo_tof.elements import PeriodicTable

pt = PeriodicTable.instance()
carbon = pt.get_element('C')

er = ElementRange(element=carbon, min_number=0, max_number=50)
er.reset()
print(f"원소: {er.element.symbol}")
print(f"범위: {er.min_number} ~ {er.max_number}")
print(f"현재: count={er.current_number}, mass={er.current_mass:.4f} Da")
print(f"min possible mass = {er.min_possible_mass:.4f} Da")
print(f"max possible mass = {er.max_possible_mass:.4f} Da")

## 2. Mass-Constrained Search 알고리즘

`FindElementalCompositions.calculate()`의 핵심 아이디어:

1. **target range** 설정: `[target_mass - tol, target_mass + tol]`
2. 모든 원소의 카운트를 최소값으로 리셋
3. **첫 번째 원소를 질량 하한에 맞게 점프** (`_init_first_element`):
   - 다른 원소들의 현재 질량 합 = `other_mass`
   - 첫 번째 원소의 필요 개수 = `(min_mass - other_mass) / mono_mass`
   - 이 점프 덕분에 불필요한 낮은 카운트를 모두 건너뛴다
4. 합산 질량이 range 안이면 **후보로 기록**
5. **odometer 방식**으로 다음 조합 탐색

```python
def _init_first_element(self) -> float:
    other_mass = sum(er.current_mass for er in self._element_ranges[1:])
    mono_mass = self._element_ranges[0].element.monoisotopic_mass
    if mono_mass > 0:
        needed = int((self._min_mass - other_mass) / mono_mass - 0.001)
        if self._element_ranges[0].current_number < needed:
            self._element_ranges[0].set_current_constrained(needed)
    return other_mass + self._element_ranges[0].current_mass
```

**최적화 포인트**: `_sort_ranges()`에서 `range/mass` 비율이 가장 큰 원소를 첫 번째로 배치한다.
H(수소)처럼 질량은 작지만 범위가 넓은 원소가 첫 번째가 되면,
점프 최적화의 효과가 극대화된다.

## 3. `FindElementalCompositions` 사용하기

클래스를 직접 생성하여 원소 범위와 electron state를 세밀하게 제어할 수 있다.

In [ ]:
from echo_tof.formula_enum import FindElementalCompositions, ElectronState

finder = FindElementalCompositions(
    min_composition="",
    max_composition="C50 H200 N10 O10 S5",
    electron_state=ElectronState.BOTH,
)

# 내부 상태 확인
print("원소 범위 (정렬 후):")
for er in finder._element_ranges:
    print(f"  {er.element.symbol:2s}: {er.min_number:3d} ~ {er.max_number:3d}  "
          f"(mono mass = {er.element.monoisotopic_mass:.6f} Da)")
print(f"\n전체 최소 질량: {finder._total_min_mass:.4f} Da")
print(f"전체 최대 질량: {finder._total_max_mass:.4f} Da")

## 4. 예제: TNT (2,4,6-trinitrotoluene) 탐색

TNT의 분자식은 **C7H5N3O6**, monoisotopic mass = **227.0178** Da.

편의 함수 `find_compositions()`로 탐색하면:

In [ ]:
from echo_tof.formula_enum import find_compositions
from echo_tof import Molecule

# TNT 이론 질량 확인
tnt = Molecule('C7H5N3O6')
print(f"TNT monoisotopic mass: {tnt.monoisotopic_mass:.6f} Da")
print(f"TNT RDB (ring + double bond): {tnt.rdb:.1f}")
print()

In [ ]:
# mass tolerance = 0.001 Da (1 mDa) 범위에서 탐색
results = find_compositions(
    target_mass=227.016,
    mass_tolerance=0.001,
    max_composition="C50 H200 N10 O10 S5",
)

print(f"후보 수: {len(results)}")
print()
for formula in results:
    mol = Molecule(formula)
    error_mda = (mol.monoisotopic_mass - 227.016) * 1000
    print(f"  {formula:20s}  mass={mol.monoisotopic_mass:.6f}  "
          f"error={error_mda:+.3f} mDa  RDB={mol.rdb:.1f}")

## 5. Tolerance에 따른 후보 수 변화

tolerance를 넓히면 후보가 급격히 증가한다. 이것이 바로 이후 filtering 단계가 필요한 이유이다.

In [ ]:
tolerances = [0.0005, 0.001, 0.005, 0.01, 0.05]
print(f"{'Tolerance (Da)':>15s}  {'Candidates':>10s}")
print("-" * 28)
for tol in tolerances:
    r = find_compositions(227.016, tol, max_composition="C50 H200 N10 O10 S5")
    print(f"{tol:>15.4f}  {len(r):>10d}")

## 6. Electron State 필터

`ElectronState`로 even-electron (짝수 전자) 또는 odd-electron (홀수 전자) 분자만 선택할 수 있다.
일반적인 [M+H]+ 이온은 even-electron이다.

```python
class ElectronState(Enum):
    BOTH = "both"         # 필터 없음
    ODD_ONLY = "odd_only" # 라디칼 이온만
    EVEN_ONLY = "even_only" # 일반 이온만
```

In [ ]:
for state in ElectronState:
    r = find_compositions(
        227.016, 0.005,
        max_composition="C50 H200 N10 O10 S5",
        electron_state=state,
    )
    print(f"{state.value:12s} -> {len(r)} candidates")

## 정리

| 단계 | 내용 |
|------|------|
| 입력 | target mass, tolerance, 원소 범위 |
| 핵심 최적화 | 첫 번째 원소 점프 (mass-constrained pruning) |
| 출력 | 분자식 문자열 리스트 |
| 한계 | tolerance가 넓으면 후보가 수천 개 → 필터링 필요 |

다음 노트북에서는 각 후보의 **isotope pattern**을 계산하는 방법을 다룬다.